**STEP 1: Simulation**

In [1]:
import numpy as np
import random
from scipy.stats import wishart
def simulate_matrix(n, d, no_components,rho,abs_bound_X):
    ###Form the X matrix
    X = np.zeros((0, d))
    # Use base_value and reminder to evenly distributed the number of samples in each component
    def generate_components(n, no_components):
        base_value = n // no_components
        remainder = n % no_components
        components = [base_value] * no_components
        for i in range(remainder):
            components[i] += 1
        random.shuffle(components)
        if sum(components) > n:
            components[n-1] -= sum(components) - n
        elif sum(components) < n:
            components[n-1] += n - sum(components)
        return components
    # Generate these multivariate gaussian components in different levels of mean and cov.
    num_list = generate_components(n, no_components)
    for i in range(no_components):
        # each level's mean value, grow by 4's power.
        mean_value = 5**(i+1)
        # 0.8 probability of 1 and 0.2 probability of 0 for creating a sparse mean vector.
        mean_vector = np.random.binomial(1, 0.8, size = d)
        mean = mean_value * mean_vector
        # randomly generate a cov matrix and make sure that it is symmetric.
        cov = wishart.rvs(df=d, scale=np.eye(d))
        # add a small value to the diagonal to ensure positive-definiteness
        cov += np.eye(d) * 1e-6
        # generate a component of X.
        X_ = np.random.multivariate_normal(mean, cov, num_list[i])
        X_ = np.array(X_)
        # append the component to the list.
        X = np.vstack((X, X_))
    X = np.array(X)
    # Set the elements < 3 to 0 to make it a sparse matrix
    X[abs(X) < abs_bound_X] = 0
    ###Form the fixed beta
    np.random.seed(39)
    beta_value = np.random.binomial(1, 0.8, size = d)
    beta = (beta_value * np.random.uniform(0, 1, size = d)).reshape(-1,1)
    beta = np.array(beta)
    np.random.seed(None)
    ###Form the error term - epsilon, which has the auto-correlation structure in time series analysis.
    r = rho # autocorrelation coefficient
    epsilon_matrix = np.zeros((0,1)) # define a zero initial matrix for vstack
    ###Form the error term in the same levels
    for i in range(no_components):
        epsilon = np.zeros((num_list[i],1))
        epsilon[0] = np.random.normal((5**(i+1))/100,1)
        for t in range(1,num_list[i]):
            epsilon[t] = r * epsilon[t-1] + np.random.normal((5**(i+1))/100,1)
        epsilon = np.array(epsilon)
        epsilon_matrix = np.vstack((epsilon_matrix, epsilon))
    epsilon_matrix = np.array(epsilon_matrix)
    epsilon = epsilon_matrix
    ###Form the Y matrix
    Y = np.array(X @ beta + epsilon)
    ###Count the number of elements < 0.1 in X and Y
    return X,Y,beta,epsilon

# Matrix Settled
n = 3000       # number of rows         
d = 200        # number of columns
no_components = 4 # number of levels of multivariate gaussian distribution in X
rho = 0.8 # autocorrelation coefficient in time series analysis
abs_bound_X = 2 # absolute bound for X to set elements to be 0
X,Y,beta,epsilon = simulate_matrix(n, d, no_components,rho,abs_bound_X)
print("The simulated matrix X is:")
print(X)
print("The simulated vector Y is:")
print(Y)

The simulated matrix X is:
[[  3.65895275  -9.240268    23.97125967 ...  13.95940101  -2.92508234
    7.15927437]
 [ 22.83213264   9.29763142   6.13872656 ... -15.62381983  10.35302409
   11.38156315]
 [-27.94554356  -5.08189151   0.         ...   4.25242478  15.48065431
   -9.99777818]
 ...
 [633.86722399 611.49337362 617.94908958 ... 633.41447074  19.85637517
    5.10052307]
 [656.73009575 622.61345714 607.84868495 ... 635.80753508   3.93276649
   11.14771086]
 [616.78085057 614.50994703 626.48635538 ... 597.3535695   -6.00032042
   -9.35823554]]
The simulated vector Y is:
[[  436.5686893 ]
 [  221.05463361]
 [  356.25115338]
 ...
 [38616.88902894]
 [38722.21494469]
 [38521.61317527]]


*Define functions for calculating ratio and norms*

In [2]:
def calculate_the_norm_square(A,b,x_selected):
    return (np.linalg.norm(A @ x_selected - b)) ** 2
def abs_error_ratio(A,b,x_hat_bar,x_star):
    return calculate_the_norm_square(A,b,x_hat_bar) - calculate_the_norm_square(A,b,x_star) / calculate_the_norm_square(A,b,x_star)

*Try to figure out what x_star should be exiquisitely*

In [3]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import lsqr
from scipy.linalg import cho_factor, cho_solve
def solver(A,b,solver):
    if solver == 'lsqr':
    # lsqr algorithm
        X = csr_matrix(A)
        Y = np.array(b)
        x_star = lsqr(X, Y)[0]
    # cholesky decomposition algorithm
    elif solver == 'cholesky':
        L,low = cho_factor(A.T@A)
        x_star = cho_solve((L,low),A.T@b)
    # Direct solver
    elif solver == 'direct':
        x_star = np.linalg.inv(A.T@A)@A.T@b

    return x_star
x_star_lsqr = solver(X,Y,'lsqr')
x_star_cholesky = solver(X,Y,'cholesky')
x_star_direct = solver(X,Y,'direct')
norm_lsqr = calculate_the_norm_square(X,Y,x_star_lsqr)
norm_cholesky = calculate_the_norm_square(X,Y,x_star_cholesky)
norm_direct = calculate_the_norm_square(X,Y,x_star_direct)
print("The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:")
print(norm_lsqr, norm_cholesky, norm_direct)
minimum_norm_square = min(norm_lsqr, norm_cholesky, norm_direct)
print("Here we output the minimum of the norm suqare for x_star calculated as:")
print(minimum_norm_square)

The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:
4382489199809352.5 8479.640907217425 8479.64090721728
Here we output the minimum of the norm suqare for x_star calculated as:
8479.64090721728


In [4]:
###From the above different norm and optimal solver calculations, it is obvious that:
###the direct solver and cholesky decomposition are the optimal solvers.
###Since their norms are very very very nearly the same
###we choose the direct solver as our optimal solver when matrix is not so large.
###And we choose the cholesky decomposition as our optimal solver when matrix is so large.
x_Star = x_star_direct
norm_Star = norm_direct
print("The optimal solver is:")
print(x_Star)

The optimal solver is:
[[ 6.49555236e-01]
 [ 7.16116102e-01]
 [ 2.75434280e-03]
 [ 3.76850486e-01]
 [ 1.97172438e-01]
 [ 6.39192107e-01]
 [ 4.93538775e-01]
 [ 1.41291312e-01]
 [ 5.17432358e-01]
 [-1.57705744e-03]
 [ 7.89194226e-04]
 [-2.11257303e-04]
 [ 5.21369766e-03]
 [ 9.17694646e-01]
 [ 3.69665980e-03]
 [-4.38765237e-04]
 [ 4.71604330e-01]
 [ 1.47369205e-01]
 [ 4.00158916e-01]
 [-1.02264105e-04]
 [ 5.21974942e-01]
 [-4.51458704e-03]
 [ 5.58959665e-02]
 [ 8.96748130e-01]
 [ 9.23926768e-01]
 [-4.92814562e-05]
 [ 7.03250514e-01]
 [ 4.69079250e-01]
 [ 1.29609635e-01]
 [ 1.99358944e-02]
 [ 2.70485951e-01]
 [ 7.49155923e-01]
 [ 1.76909711e-01]
 [-2.80164375e-03]
 [ 6.12437989e-02]
 [ 5.51928315e-01]
 [ 1.13320028e-01]
 [ 1.65336285e-03]
 [ 3.11183344e-02]
 [ 3.29965423e-03]
 [ 2.38711630e-01]
 [ 5.89511323e-01]
 [ 6.21303931e-01]
 [-3.75220988e-03]
 [ 9.00545732e-01]
 [ 5.28136156e-01]
 [ 5.30422372e-01]
 [ 6.19375674e-01]
 [ 6.48456791e-01]
 [ 4.76524546e-01]
 [ 5.33888716e-02]
 [ 9.054

**STEP 2: Uniform Sampling sketch // Algorithm 1: Distributed Randomized Regression**

In [5]:
# e.g. our desired sketching size is m = 1000
from operator import index


m = 1000

# Algorithm 1 inserting inside uniform sampling: Distributed Randomized Regression

# S_k @ A here is just computed as A [uniformly_sampled_index]
# As S_k here is just a diagnoal matrix of 1 or 0 where sampled rows have 1 as value
def uniform_sampling(A,b,n,m,q):
    x_hat_list = []
    for i in range(q):
        index = np.random.choice(n, size=m, replace=False)
        A_sk = A[index]
        b_sk = b[index]
        x_hat = np.linalg.inv(A_sk.T @ A_sk) @ A_sk.T @ b_sk
        x_hat_list.append(x_hat)
    x_bar = sum(x_hat_list) / q
    return x_bar
